# Pipeline GPU 1 : PyTorch -> ONNX -> TensorRT -> CUDA/cuDNN -> GPU
Runtime : GPU T4. FP32 puis FP16.

In [1]:
!nvidia-smi
!pip -q install tensorrt onnx onnxscript nvidia-ml-py

Tue Nov  3 16:21:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import sys, os
sys.path.append('/content/drive/MyDrive/tp_deploy')
from common import *
X, y = load_data()
print(X.shape, y[:10])

Mounted at /content/drive
torch.Size([50, 3, 224, 224]) [2 1 5 8 9 3 8 9 0 1]


In [3]:
model = load_model()
torch.onnx.export(model, torch.randn(BS,3,224,224), 'resnet18.onnx', input_names=['input'],
                  output_names=['logits'], opset_version=17, dynamo=False)
import tensorrt as trt
logger = trt.Logger(trt.Logger.WARNING)

def build_engine(fp16):
    builder = trt.Builder(logger)
    network = builder.create_network(0)
    parser = trt.OnnxParser(network, logger)
    assert parser.parse(open('resnet18.onnx', 'rb').read()), [parser.get_error(i) for i in range(parser.num_errors)]
    config = builder.create_builder_config()
    if fp16:
        config.set_flag(trt.BuilderFlag.FP16)
    t = time.perf_counter()
    blob = builder.build_serialized_network(network, config)
    print('build engine fp16=%s : %.1f s' % (fp16, time.perf_counter() - t))
    return trt.Runtime(logger).deserialize_cuda_engine(blob)

def make_predict(engine):
    ctx = engine.create_execution_context()
    inp = torch.empty(BS, 3, 224, 224, device='cuda')
    out = torch.empty(BS, 10, device='cuda')
    ctx.set_tensor_address('input', inp.data_ptr())
    ctx.set_tensor_address('logits', out.data_ptr())
    stream = torch.cuda.current_stream()
    def predict(b):
        inp.copy_(b)
        ctx.execute_async_v3(stream.cuda_stream)
        stream.synchronize()
        return out.cpu().numpy()
    return predict

/tmp/ipykernel_418/1616244395.py:2: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, torch.randn(BS,3,224,224), 'resnet18.onnx', input_names=['input'],


In [4]:
res32 = run_benchmark('gpu_tensorrt_fp32', make_predict(build_engine(False)), X, y, device='gpu')

build engine fp16=False : 8.2 s
total_s      moy=0.0436 std=0.0001 min=0.0435 max=0.0437
throughput   moy=1145.7618 std=2.5753 min=1142.9136 max=1149.2797
batch_ms     moy=8.7185 std=0.0197 min=8.6915 max=8.7396
top1         moy=0.9200 std=0.0000 min=0.9200 max=0.9200
énergie : {'watts': 65.81782133333333, 'joules_per_image': 0.06012966950717214}
